# Agent RNE Expert V-PDF / Gidoo


✅ **V9 (enrichie)** :
- Renommage intelligent des fichiers via Mistral AI + nomenclature INPI (PJ_xx)
- Extraction texte PDF (PyMuPDF + pypdf fallback)
- Analyse documentaire Mistral pour chaque document
- Analyse RNE des formalités (inputs API INPI) via Mistral
- Rapport PDF A4 paysage Gidoo (cover, sommaire hiérarchique, fiche entreprise structurée, documents, points d'attention, clôture)
- Structure ZIP inchangée : `COMPTES_ANNUELS/` + `ACTES/` + `RAPPORT/`

✅ **V8** : Correction nommage (dénomination + dates 0000-00-00 ignorées). Validation formulaire contact.


## 1️⃣ Installation (Python libs)

In [ ]:
!pip install flask requests pymupdf pypdf reportlab -q
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
print("✅ Prêt")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 34.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.4/331.4 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 27.7 MB/s eta 0:00:00
✅ Prêt


## 2️⃣ Configuration (identifiants INPI + options)

In [ ]:
INPI_USERNAME = "ask.me.spc@gmail.com"
INPI_PASSWORD = "5Z-36Cg+g$*LJw."

# Email uniquement pour le formulaire "Contactez-nous" (Gmail SMTP)
SMTP_HOST     = "smtp.gmail.com"
SMTP_PORT     = 587
SMTP_USER     = "sebastien.muff@gmail.com"
SMTP_PASSWORD = "twox aakp frep mzse"   # ⚠️ REMPLACER par le Mot de Passe d'Application Google (16 car.)
CONTACT_EMAIL = "sebastien.muff@gmail.com"

PORT = 5000

print("✅ Configuration chargée")
print(f"   🔐 INPI : {INPI_USERNAME}")
print(f"   📩 Contact -> {CONTACT_EMAIL}")

# ── Mistral AI (renommage + analyse) ──
MISTRAL_API_KEY = "uKUyeQgQLZEHeq2WuEiCZOuqDYl8lvIh"
MISTRAL_URL = "https://api.mistral.ai/v1/chat/completions"
MISTRAL_MODEL = "mistral-small-latest"
MISTRAL_TEMPERATURE = 0.2
MISTRAL_MAX_TOKENS_NAMING = 140
MISTRAL_MAX_TOKENS_ANALYSE = 800
MISTRAL_SLEEP_BETWEEN_CALLS = 10
MISTRAL_RETRIES_429 = 3

print(f"   🤖 Mistral : {MISTRAL_MODEL}")


✅ Configuration chargée
   🔐 INPI : ask.me.spc@gmail.com
   📩 Contact -> sebastien.muff@gmail.com
   🤖 Mistral : mistral-small-latest


## 3️⃣ Application (Flask)

✅ Le HTML est intégré dans `HTML_PAGE` et servi sur `/`.

✅ **Aucun email n'est envoyé à la fin de la collecte.**

📩 Le seul email est celui du formulaire **Contactez-nous** (SMTP Gmail).

✅ **V8** : Validation email + tél mobile. Nommage : dates `0000-00-00` ignorées, dénomination propagée.

In [ ]:
import os, io, json, time, uuid, zipfile, logging, smtplib, ssl, re, base64 as _b64
import fitz  # PyMuPDF
from typing import Any, Dict, List
from datetime import datetime
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
from threading import Thread
import requests as http_requests
from flask import Flask, request as flask_request, jsonify, send_file

# -- PDF report (reportlab) --
from reportlab.lib.pagesizes import A4, landscape
from reportlab.lib.units import mm, cm
from reportlab.lib.colors import HexColor, white, black
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.enums import TA_LEFT, TA_CENTER, TA_JUSTIFY
from reportlab.platypus import (
    BaseDocTemplate, Frame, PageTemplate, NextPageTemplate,
    Paragraph as RLPara, Spacer, Table, TableStyle, PageBreak,
    Image as RLImage, HRFlowable, Flowable, KeepTogether,
)

app = Flask(__name__)
app.config['SECRET_KEY'] = 'gidoo-' + str(uuid.uuid4())[:8]
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)
jobs = {}

class Job:
    def __init__(self, siren):
        self.id = str(uuid.uuid4())[:8]
        self.siren = siren
        self.status, self.progress = "pending", 0
        self.steps, self.errors = [], []
        self.bilans_count = self.actes_count = 0
        self.zip_data = None
        self.denomination = ""
        self.total_docs = 0
        self.estimated_minutes = 0
        self.current_doc_name = ""
        self.current_doc_desc = ""
    def log(self, msg, progress=None):
        if progress is not None: self.progress = progress
        self.steps.append({"time": datetime.now().isoformat(), "message": msg, "progress": self.progress})
        logger.info(f"[{self.id}] {msg}")
    def to_dict(self):
        return {"id": self.id, "siren": self.siren, "status": self.status,
                "progress": self.progress, "steps": self.steps, "errors": self.errors,
                "bilans_count": self.bilans_count, "actes_count": self.actes_count,
                "denomination": self.denomination,
                "total_docs": self.total_docs, "estimated_minutes": self.estimated_minutes,
                "current_doc_name": self.current_doc_name, "current_doc_desc": self.current_doc_desc}

INPI = "https://registre-national-entreprises.inpi.fr/api"

def inpi_login():
    r = http_requests.post(f"{INPI}/sso/login", json={"username": INPI_USERNAME, "password": INPI_PASSWORD},
        headers={"Content-Type": "application/json"}, timeout=30)
    r.raise_for_status()
    t = r.json().get("token")
    if not t: raise ValueError("Token INPI non recu")
    return t

def inpi_get_attachments(siren, token):
    r = http_requests.get(f"{INPI}/companies/{siren}/attachments",
        headers={"Authorization": f"Bearer {token}"}, timeout=30)
    r.raise_for_status()
    return r.json()

def inpi_get_acte_info(acte_id, token):
    try:
        r = http_requests.get(f"{INPI}/actes/{acte_id}",
            headers={"Authorization": f"Bearer {token}"}, timeout=30)
        r.raise_for_status()
        return r.json().get("data", {})
    except:
        return {}

def inpi_dl(endpoint, token):
    r = http_requests.get(f"{INPI}/{endpoint}",
        headers={"Authorization": f"Bearer {token}"}, timeout=60)
    r.raise_for_status()
    return r.content

def sanitize(name):
    return re.sub(r'\s+', ' ', re.sub(r'[\\/:*?"<>|;]', '', name)).strip()

def bilan_name(b, siren, job_denomination=""):
    c = (b.get("dateCloture") or "0000-00-00")[:10]
    n = b.get("denomination") or job_denomination or "Entreprise"
    return sanitize(f"{c} - {siren} - {n} - COMPTES ANNUELS")

# Table de correspondance PJ -> libellé officiel
PJ_LABELS = {
    "PJ_01": "Copie des statuts",
    "PJ_02": "Copie des statuts mis a jour",
    "PJ_03": "Actes de nomination des organes de gestion",
    "PJ_04": "Rapport du commissaire aux apports",
    "PJ_05": "Decision des fondateurs - evaluation des apports",
    "PJ_06": "Certificat du depositaire des fonds",
    "PJ_07": "PV de l assemblee constitutive",
    "PJ_34": "Acte d acquisition par donation",
    "PJ_35": "Acte de notoriete - devolution successorale",
    "PJ_52": "Decision de modification",
    "PJ_54": "PV de modification",
    "PJ_55": "Decision du CA ou Directoire - modification capital",
    "PJ_56": "Certificat du depositaire - modification capital",
    "PJ_61": "Contrat d appui au projet d entreprise",
    "PJ_62": "Attestation information conjoint - biens communs",
    "PJ_65": "Declaration d insaisissabilite",
    "PJ_68": "Accord expres et information prealable du conjoint",
    "PJ_69": "Accord expres et information prealable du coindivisaire",
    "PJ_74": "Decision d inscription ou nomination",
    "PJ_78": "Contrat d appui EURL",
    "PJ_79": "Contrat de groupement",
    "PJ_82": "Compte de liquidation",
    "PJ_83": "Acte de cession de fonds de commerce",
    "PJ_85": "Declaration de regularite et de conformite",
    "PJ_86": "Projet de traite de fusion",
    "PJ_87": "Realisation de l apport partiel d actif",
    "PJ_91": "Rapport commissaire a la transformation",
    "PJ_92": "Certificat de depot de fonds",
    "PJ_93": "PV AG de designation des administrateurs",
    "PJ_94": "Declaration d option du conjoint collaborateur",
    "PJ_97": "Liste des sieges sociaux anterieurs",
    "PJ_102": "Bilan actif-passif",
    "PJ_103": "Compte de resultat",
    "PJ_104": "Annexe aux comptes annuels",
    "PJ_106": "Proposition d affectation du resultat",
    "PJ_107": "Resolution d affectation",
    "PJ_108": "Deliberation de l assemblee ou decision de l associe unique",
    "PJ_109": "Rapport de gestion",
    "PJ_110": "Rapport de gouvernance",
    "PJ_121": "Rapport du conseil de surveillance",
    "PJ_122": "Bilan consolide",
    "PJ_123": "Compte de resultat consolide",
    "PJ_124": "Annexe consolide",
    "PJ_125": "Rapport de gestion de groupe",
    "PJ_126": "Rapport du president - controle interne et gouvernement d entreprise",
    "PJ_127": "Tableau recapitulatif des delegations",
    "PJ_128": "Tableau resultats des 5 derniers exercices",
    "PJ_129": "Rapport CAC sur CA",
    "PJ_130": "Rapport CAC comptes consolides",
    "PJ_131": "Rapport CAC sur rapport de gouvernance",
    "PJ_133": "PV de cloture et de liquidation",
    "PJ_134": "PV de cloture et de liquidation avec enregistrement fiscal",
    "PJ_135": "PV AGE approbation fusion",
    "PJ_142": "Traduction de statut",
    "PJ_144": "Comptes combines",
    "PJ_145": "Rapport paiements aux autorites extractives",
    "PJ_146": "Declaration de confidentialite",
    "PJ_147": "Declaration de publicite simplifiee",
    "PJ_148": "Declaration de confidentialite du rapport CAC",
    "PJ_149": "Declaration de confidentialite du compte de resultat",
    "PJ_150": "Declaration d affectation du patrimoine",
    "PJ_151": "Compte de resultat de publication simplifiee",
    "PJ_152": "PV de mise a jour des statuts",
    "PJ_153": "PV de transformation",
    "PJ_154": "Bilan de publication simplifiee",
    "PJ_155": "Decision d augmentation du capital - PV AG",
    "PJ_156": "PV AG reduction de capital",
    "PJ_158": "Projet de transformation",
    "PJ_159": "Acte de nomination du commissaire a la transformation",
    "PJ_160": "Rapport du commissaire a la transformation",
    "PJ_161": "Projet de fusion",
    "PJ_163": "Rapport du commissaire aux apports",
    "PJ_164": "Projet commun de fusion transfrontaliere",
    "PJ_169": "Projet commun de fusion transfrontaliere",
    "PJ_170": "Statuts de la societe issue de la fusion transfrontaliere",
    "PJ_173": "Projet d apport partiel d actif",
    "PJ_175": "PV AG approbation projet de fusion",
    "PJ_177": "Contrat d appui",
    "PJ_178": "Declaration d information donnee au conjoint",
    "PJ_179": "Declaration d affectation du patrimoine",
    "PJ_180": "Attestation de depot des fonds",
    "PJ_183": "Requete ou ordonnance du juge commis RCS",
    "PJ_185": "Decision de transfert siege et transformation en societe de droit francais",
    "PJ_189": "Acte notarie modification repartition parts sociales",
    "PJ_190": "Acte sous seing prive modification repartition parts sociales",
    "PJ_192": "Avenant au projet de fusion",
    "PJ_193": "Avenant au projet de scission",
    "PJ_194": "Ordonnance",
    "PJ_195": "PV AG non recours commissaire aux apports",
    "PJ_196": "PV AG non approbation projet scission ou fusion",
    "PJ_197": "PV AG principe du projet scission ou fusion",
    "PJ_198": "PV AG refus approbation comptes annuels",
    "PJ_199": "Projet de scission",
    "PJ_200": "Projet de statuts mis a jour",
    "PJ_201": "Projet de traite d apport",
    "PJ_202": "Rapport du commissaire a la fusion",
    "PJ_203": "Rapport du commissaire a la scission",
    "PJ_204": "Rapport du liquidateur",
    "PJ_227": "Comptes sociaux",
    "PJ_231": "Bilan et compte de resultat",
    "PJ_232": "Bilan compte de resultat et annexes",
    "PJ_233": "Bilan et annexes",
    "PJ_234": "Bilan simplifie et annexes",
    "PJ_235": "Rapport CAC",
    "PJ_236": "PV d assemblee - affectation du resultat",
    "PJ_240": "Attestation fiscale",
    "PJ_241": "Attestation de regularite sociale",
    "PJ_243": "Copie d un jugement diffusible",
}

def resolve_nature(ai, ab):
    """Determine la nature de l acte en priorite via typeDocument (code PJ), puis typeRdd."""
    # 1) typeDocument = code PJ_XX => on cherche dans la table
    td = (ai.get("typeDocument") or ab.get("typeDocument") or "").strip()
    if td:
        # Normaliser : "pj_233" -> "PJ_233"
        td_upper = td.upper().replace(" ", "_")
        if td_upper in PJ_LABELS:
            return PJ_LABELS[td_upper]

    # 2) typeRdd => typeActe + decision
    tr = ai.get("typeRdd") or ab.get("typeRdd") or []
    if tr and isinstance(tr, list) and len(tr) > 0:
        e = tr[0]
        ta = (e.get("typeActe") or "").strip()
        dec = (e.get("decision") or "").strip()
        combined = f"{ta} - {dec}" if ta and dec else (ta or dec)
        if combined:
            return combined

    # 3) Fallback sur typeDocument brut (non-PJ)
    if td:
        return td

    return ""

def _valid_date(*candidates):
    """Retourne la premiere date valide (non vide, non 0000-00-00)."""
    for d in candidates:
        if d and isinstance(d, str) and d[:10].replace("-","").strip("0") != "":
            return d[:10]
    return "0000-00-00"

def acte_name(ai, ab, siren, job_denomination=""):
    d = _valid_date(
        ai.get("dateDepot"), ab.get("dateDepot"),
        ai.get("dateDepotFormate"), ab.get("dateDepotFormate"),
        ai.get("dateCreation"), ab.get("dateCreation"),
        ai.get("dateActe"), ab.get("dateActe"),
    )
    n = ai.get("denomination") or ab.get("denomination") or job_denomination or "Entreprise"
    nature = resolve_nature(ai, ab)
    base = f"{d} - {siren} - {n}"
    if nature:
        return sanitize(f"{base} - {nature}")
    return sanitize(base)

def send_contact_email(name, email, phone, message):
    """Envoi SMTP Gmail (STARTTLS:587) pour le formulaire Contactez-nous."""
    subject = f"[Gidoo] Demande d'intégration — {name}"
    body = (
        "Nouvelle demande via le formulaire Contactez-nous\n"
        "----------------------------------------------\n"
        f"Nom      : {name}\n"
        f"Email    : {email}\n"
        f"Téléphone: {phone}\n\n"
        "Message :\n"
        f"{message}\n"
    )

    msg = MIMEMultipart()
    msg["From"] = SMTP_USER
    msg["To"] = CONTACT_EMAIL
    msg["Reply-To"] = email
    msg["Subject"] = subject
    msg.attach(MIMEText(body, "plain", "utf-8"))

    try:
        with smtplib.SMTP(SMTP_HOST, SMTP_PORT, timeout=15) as srv:
            srv.ehlo()
            srv.starttls(context=ssl.create_default_context())
            srv.ehlo()
            srv.login(SMTP_USER, SMTP_PASSWORD)
            srv.sendmail(SMTP_USER, [CONTACT_EMAIL], msg.as_string())
        logger.info(f"[CONTACT SENT] from={email} name={name} phone={phone}")
        return True
    except Exception as e:
        logger.error(f"[CONTACT FAILED] {e}")
        return False

# ═══════════════════════════════════════════
# INPI Nomenclature complète (typeDocument -> libellé officiel)
# ═══════════════════════════════════════════
INPI_TYPE_DOCUMENT_MAP = {
    "PJ_01": "Copie des statuts",
    "PJ_02": "Copie des statuts mis à jour",
    "PJ_03": "Actes de nomination des organes de gestion",
    "PJ_04": "Rapport du commissaire aux apports",
    "PJ_05": "Décision des fondateurs - évaluation des apports",
    "PJ_06": "Certificat du dépositaire des fonds",
    "PJ_07": "PV de l assemblée constitutive",
    "PJ_34": "Acte d acquisition par donation",
    "PJ_35": "Acte de notoriété - dévolution successorale",
    "PJ_52": "Décision de modification",
    "PJ_54": "PV ayant décidé et constaté la modification",
    "PJ_55": "Décision du CA ou Directoire - modification capital",
    "PJ_56": "Certificat du dépositaire - modification capital",
    "PJ_61": "Contrat d appui au projet d entreprise",
    "PJ_62": "Attestation information conjoint - biens communs",
    "PJ_65": "Déclaration d insaisissabilité",
    "PJ_68": "Accord exprès et information préalable du conjoint",
    "PJ_69": "Accord exprès et information préalable du coindivisaire",
    "PJ_74": "Décision d inscription ou nomination",
    "PJ_78": "Contrat d appui EURL",
    "PJ_79": "Contrat de groupement",
    "PJ_82": "Compte de liquidation",
    "PJ_83": "Acte de cession de fonds de commerce",
    "PJ_85": "Déclaration de régularité et de conformité",
    "PJ_86": "Projet de traité de fusion",
    "PJ_87": "Réalisation de l apport partiel d actif",
    "PJ_91": "Rapport commissaire à la transformation",
    "PJ_92": "Certificat de dépôt de fonds",
    "PJ_93": "PV AG de désignation des administrateurs",
    "PJ_94": "Déclaration d option du conjoint collaborateur",
    "PJ_97": "Liste des sièges sociaux antérieurs",
    "PJ_102": "Bilan actif-passif",
    "PJ_103": "Compte de résultat",
    "PJ_104": "Annexe aux comptes annuels",
    "PJ_106": "Proposition d affectation du résultat",
    "PJ_107": "Résolution d affectation",
    "PJ_108": "Délibération de l assemblée ou décision de l associé unique",
    "PJ_109": "Rapport de gestion",
    "PJ_110": "Rapport de gouvernance",
    "PJ_121": "Rapport du conseil de surveillance",
    "PJ_122": "Bilan consolidé",
    "PJ_123": "Compte de résultat consolidé",
    "PJ_124": "Annexe consolidé",
    "PJ_125": "Rapport de gestion de groupe",
    "PJ_126": "Rapport du président - contrôle interne et gouvernement d entreprise",
    "PJ_127": "Tableau récapitulatif des délégations",
    "PJ_128": "Tableau résultats des 5 derniers exercices",
    "PJ_129": "Rapport CAC sur CA",
    "PJ_130": "Rapport CAC comptes consolidés",
    "PJ_131": "Rapport CAC sur rapport de gouvernance",
    "PJ_133": "PV de clôture et de liquidation",
    "PJ_134": "PV de clôture et de liquidation avec enregistrement fiscal",
    "PJ_135": "PV AGE approbation fusion",
    "PJ_142": "Traduction de statut",
    "PJ_144": "Comptes combinés",
    "PJ_145": "Rapport paiements aux autorités extractives",
    "PJ_146": "Déclaration de confidentialité",
    "PJ_147": "Déclaration de publicité simplifiée",
    "PJ_148": "Déclaration de confidentialité du rapport CAC",
    "PJ_149": "Déclaration de confidentialité du compte de résultat",
    "PJ_150": "Déclaration d affectation du patrimoine",
    "PJ_151": "Compte de résultat de publication simplifiée",
    "PJ_152": "PV de mise à jour des statuts",
    "PJ_153": "PV de transformation",
    "PJ_154": "Bilan de publication simplifiée",
    "PJ_155": "Décision d augmentation du capital - PV AG",
    "PJ_156": "PV AG réduction de capital",
    "PJ_158": "Projet de transformation",
    "PJ_159": "Acte de nomination du commissaire à la transformation",
    "PJ_160": "Rapport du commissaire à la transformation",
    "PJ_161": "Projet de fusion",
    "PJ_163": "Rapport du commissaire aux apports",
    "PJ_164": "Projet commun de fusion transfrontalière",
    "PJ_169": "Projet commun de fusion transfrontalière",
    "PJ_170": "Statuts de la société issue de la fusion transfrontalière",
    "PJ_173": "Projet d apport partiel d actif",
    "PJ_175": "PV AG approbation projet de fusion",
    "PJ_177": "Contrat d appui",
    "PJ_178": "Déclaration d information donnée au conjoint",
    "PJ_179": "Déclaration d affectation du patrimoine",
    "PJ_180": "Attestation de dépôt des fonds",
    "PJ_183": "Requête ou ordonnance du juge commis RCS",
    "PJ_185": "Décision de transfert siège et transformation en société de droit français",
    "PJ_189": "Acte notarié modification répartition parts sociales",
    "PJ_190": "Acte sous seing privé modification répartition parts sociales",
    "PJ_192": "Avenant au projet de fusion",
    "PJ_193": "Avenant au projet de scission",
    "PJ_194": "Ordonnance",
    "PJ_195": "PV AG non recours commissaire aux apports",
    "PJ_196": "PV AG non approbation projet scission ou fusion",
    "PJ_197": "PV AG principe du projet scission ou fusion",
    "PJ_198": "PV AG refus approbation comptes annuels",
    "PJ_199": "Projet de scission",
    "PJ_200": "Projet de statuts mis à jour",
    "PJ_201": "Projet de traité d apport",
    "PJ_202": "Rapport du commissaire à la fusion",
    "PJ_203": "Rapport du commissaire à la scission",
    "PJ_204": "Rapport du liquidateur",
    "PJ_227": "Comptes sociaux",
    "PJ_231": "Bilan et compte de résultat",
    "PJ_232": "Bilan compte de résultat et annexes",
    "PJ_233": "Bilan et annexes",
    "PJ_234": "Bilan simplifié et annexes",
    "PJ_235": "Rapport CAC",
    "PJ_236": "PV d assemblée - affectation du résultat",
    "PJ_240": "Attestation fiscale",
    "PJ_241": "Attestation de régularité sociale",
    "PJ_243": "Copie d un jugement diffusible",
}

def type_doc_label(code):
    return INPI_TYPE_DOCUMENT_MAP.get((code or "").upper().replace(" ", "_"), "")

# ═══════════════════════════════════════════
# PDF -> TEXTE (PyMuPDF + pypdf fallback)
# ═══════════════════════════════════════════
def pdf_bytes_to_text(pdf_bytes):
    if not pdf_bytes:
        return ""
    try:
        with fitz.open(stream=pdf_bytes, filetype="pdf") as doc:
            parts = [page.get_text("text") or "" for page in doc]
            out = "\n\n".join(p for p in parts if p.strip())
            if out.strip():
                return out.strip()
    except Exception:
        pass
    try:
        from pypdf import PdfReader
        reader = PdfReader(io.BytesIO(pdf_bytes))
        parts = [p.extract_text() or "" for p in reader.pages]
        return "\n\n".join(p for p in parts if p.strip()).strip()
    except Exception:
        return ""

# ═══════════════════════════════════════════
# Mistral AI client (retry 429)
# ═══════════════════════════════════════════
def mistral_chat_once(prompt, max_tokens=512):
    payload = {
        "model": MISTRAL_MODEL,
        "messages": [{"role": "user", "content": prompt}],
        "temperature": MISTRAL_TEMPERATURE,
        "max_tokens": max_tokens,
        "top_p": 1,
        "response_format": {"type": "text"},
        "safe_prompt": False,
    }
    r = http_requests.post(
        MISTRAL_URL,
        headers={"Authorization": f"Bearer {MISTRAL_API_KEY}", "Content-Type": "application/json"},
        json=payload,
        timeout=120,
    )
    if r.status_code == 429:
        raise RuntimeError("Mistral 429")
    r.raise_for_status()
    return r.json()["choices"][0]["message"]["content"]

def mistral_chat(prompt, max_tokens=512):
    last_err = None
    for attempt in range(1, MISTRAL_RETRIES_429 + 1):
        try:
            return mistral_chat_once(prompt, max_tokens=max_tokens)
        except Exception as e:
            last_err = e
            if "429" in str(e):
                wait = MISTRAL_SLEEP_BETWEEN_CALLS * attempt
                logger.info(f"Mistral 429 -> attente {wait}s (tentative {attempt}/{MISTRAL_RETRIES_429})")
                time.sleep(wait)
            else:
                raise
    raise last_err

# ═══════════════════════════════════════════
# Prompts Mistral
# ═══════════════════════════════════════════
PROMPT_NOMMAGE = (
    "###ROLE\n"
    "Tu nommes des documents juridiques INPI RNE.\n"
    "\n"
    "###OBJECTIF\n"
    "Produire UNIQUEMENT un nom de fichier (UNE SEULE LIGNE) au format exact :\n"
    "{{DATE_DEPOT}} - {{SIREN}} - {{DENOMINATION}} - {{FAMILLE}} - {{LIBELLE}}\n"
    "\n"
    "REGLES ABSOLUES\n"
    "- 1 ligne, aucun texte avant/apres.\n"
    "- Pas d'extension (.pdf/.txt).\n"
    "- Pas de guillemets, pas de point final.\n"
    "- Pas de caracteres interdits.\n"
    "- Pas de redondance (ne repete pas SIREN / denomination / adresse / forme juridique).\n"
    "- {{LIBELLE}} = 3 a 10 mots maximum.\n"
    "- IMPORTANT : si typeDocument est present et a un libelle officiel, utilise-le comme base du libelle.\n"
    "\n"
    "DONNEES\n"
    "SIREN: {SIREN}\n"
    "DENOMINATION: {DENOMINATION}\n"
    "DATE_DEPOT: {DATE_DEPOT}\n"
    "FAMILLE: {FAMILLE}\n"
    "\n"
    "TYPE INPI\n"
    "typeDocument: {TYPE_CODE}\n"
    "libelle officiel: {TYPE_LIBELLE}\n"
    "\n"
    "META (brut):\n"
    "{META}\n"
    "\n"
    "TEXTE EXTRAIT (peut etre vide):\n"
    "{DOC_TEXT}\n"
)

PROMPT_ANALYSE = (
    "Tu es un assistant d'analyse documentaire juridique.\n"
    "\n"
    "OBJECTIF\n"
    "Extraire UNIQUEMENT les informations specifiques au document (fidele, factuel).\n"
    "Si typeDocument + libelle officiel existent, ils servent de contexte, mais les faits doivent venir du texte.\n"
    "\n"
    "REGLES ABSOLUES\n"
    "- Ne jamais rappeler SIREN / denomination / adresse / forme juridique.\n"
    "- Ne rien inventer.\n"
    "- Si texte vide/inexploitable -> NON analyse.\n"
    "\n"
    "FORMAT (exactement)\n"
    "\n"
    "Document analyse :\n"
    "OUI ou NON\n"
    "\n"
    "Nature INPI :\n"
    "{TYPE_CODE} - {TYPE_LIBELLE}\n"
    "\n"
    "Descriptif :\n"
    "- Si OUI : 1 paragraphe factuel, uniquement contenu du document.\n"
    "- Si NON : ecrire exactement :\n"
    "Aucun contenu textuel exploitable n'a pu etre extrait du document.\n"
    "\n"
    "TEXTE EXTRAIT :\n"
    "{DOC_TEXT}\n"
)

PROMPT_RNE_ANALYSE = (
    "Tu es un expert en analyse de donnees RNE (Registre National des Entreprises) de l'INPI.\n"
    "\n"
    "OBJECTIF\n"
    "Analyser les donnees d'entreprise issues de l'API INPI RNE et produire une synthese structuree.\n"
    "\n"
    "REGLES\n"
    "- Factuel uniquement, aucune invention.\n"
    "- Format clair et lisible.\n"
    "- Mentionne les points cles : identite, forme juridique, dirigeants, capital, adresse, activite (NAF), dates cles, etablissements.\n"
    "- Signale toute anomalie ou point d'attention.\n"
    "\n"
    "DONNEES ENTREPRISE (JSON) :\n"
    "{COMPANY_JSON}\n"
    "\n"
    "FORMALITES / ATTACHMENTS :\n"
    "- {NB_BILANS} bilan(s)\n"
    "- {NB_ACTES} acte(s)\n"
)

# ═══════════════════════════════════════════
# Helpers analyse
# ═══════════════════════════════════════════
def parse_analysis(text):
    analysed = "NON"
    desc = ""
    m = re.search(r"Document analys[eé]\s*:\s*(OUI|NON)", text, flags=re.IGNORECASE)
    if m:
        analysed = m.group(1).upper()
    m2 = re.search(r"Descriptif\s*:\s*(.*)", text, flags=re.IGNORECASE|re.DOTALL)
    if m2:
        desc = m2.group(1).strip()
    return {"analysed": analysed, "descriptif": desc}

def _safe_filename(name, max_len=180):
    name = re.sub(r'[\\/:*?"<>|;]+', " ", str(name)).strip()
    name = re.sub(r"\s+", " ", name)
    return (name[:max_len].rstrip() if len(name) > max_len else name) or "document"

def process_one_document(famille, meta, pdf_bytes, siren, denomination, job):
    """Traite un document : extraction texte + renommage Mistral + analyse Mistral."""
    date_depot = _valid_date(
        meta.get("dateDepot"), meta.get("dateDepotFormate"),
        meta.get("updatedAt"), meta.get("createdAt"), meta.get("dateActe"),
    )
    doc_text = pdf_bytes_to_text(pdf_bytes)
    short_text = doc_text[:16000] if doc_text else ""

    type_code = (meta.get("typeDocument") or "").strip()
    type_lib = type_doc_label(type_code)

    # Renommage via Mistral
    naming_prompt = PROMPT_NOMMAGE.format(
        DATE_DEPOT=date_depot,
        SIREN=siren,
        DENOMINATION=denomination,
        FAMILLE=famille,
        TYPE_CODE=type_code or "N/A",
        TYPE_LIBELLE=type_lib or "N/A",
        META=json.dumps(meta, ensure_ascii=False)[:2500],
        DOC_TEXT=short_text,
    )
    try:
        filename_base = mistral_chat(naming_prompt, max_tokens=MISTRAL_MAX_TOKENS_NAMING).splitlines()[0].strip()
        filename_base = _safe_filename(filename_base)
    except Exception as e:
        logger.warning(f"Mistral naming error: {e}")
        if famille == "COMPTES ANNUELS":
            filename_base = bilan_name(meta, siren, denomination)
        else:
            filename_base = acte_name(meta, meta, siren, denomination)

    # Analyse via Mistral
    analysis_prompt = PROMPT_ANALYSE.format(
        TYPE_CODE=type_code or "N/A",
        TYPE_LIBELLE=type_lib or "N/A",
        DOC_TEXT=(doc_text[:20000] if doc_text else ""),
    )
    try:
        analysis_raw = mistral_chat(analysis_prompt, max_tokens=MISTRAL_MAX_TOKENS_ANALYSE)
        parsed = parse_analysis(analysis_raw)
    except Exception as e:
        logger.warning(f"Mistral analysis error: {e}")
        analysis_raw = "Erreur analyse Mistral"
        parsed = {"analysed": "NON", "descriptif": "Erreur lors de l'analyse Mistral."}

    time.sleep(MISTRAL_SLEEP_BETWEEN_CALLS)

    return {
        "filename_base": filename_base,
        "date_depot": date_depot,
        "typeDocument": type_code,
        "typeLibelle": type_lib,
        "txt": doc_text,
        "analysis_raw": analysis_raw,
        "document_analyse": parsed["analysed"],
        "descriptif": parsed["descriptif"],
        "famille": famille,
    }

# ══════════════════════════════════════
PAGE = landscape(A4)  # 297 x 210 mm
PW, PH = PAGE

# Brand palette
# ══════════════════════════════════════
C_ORANGE  = HexColor("#EF8829")
C_TEAL    = HexColor("#017e84")
C_NAVY    = HexColor("#1a2744")
C_NAVY_L  = HexColor("#243456")
C_TXT     = HexColor("#2c2c2c")
C_MUTED   = HexColor("#7a7470")
C_BORDER  = HexColor("#e8e4df")
C_LGRAY   = HexColor("#f3f1ee")
C_CARD    = HexColor("#ffffff")
C_RED     = HexColor("#c0392b")
C_GREEN   = HexColor("#27ae60")
C_ALERT_BG = HexColor("#fff8f0")

# ══════════════════════════════════════
# Logo
# ══════════════════════════════════════
_LOGO_B64 = "..."

def _logo(w_cm=4):
    b = _b64.b64decode(_LOGO_B64)
    return RLImage(io.BytesIO(b), width=w_cm*cm, height=w_cm*cm*0.50)

# ══════════════════════════════════════
# Styles
# ══════════════════════════════════════
def _styles():
    s = getSampleStyleSheet()
    a = s.add
    # Cover
    a(ParagraphStyle("CvT", fontName="Helvetica-Bold", fontSize=34, leading=42, textColor=white, alignment=TA_LEFT))
    a(ParagraphStyle("CvS", fontName="Helvetica", fontSize=15, leading=22, textColor=HexColor("#b0c4de"), alignment=TA_LEFT))
    a(ParagraphStyle("CvM", fontName="Helvetica", fontSize=11, leading=16, textColor=HexColor("#8899bb"), alignment=TA_LEFT))
    # TOC
    a(ParagraphStyle("TocT", fontName="Helvetica-Bold", fontSize=22, leading=28, textColor=C_NAVY, spaceAfter=16))
    a(ParagraphStyle("Toc1", fontName="Helvetica-Bold", fontSize=11, leading=20, textColor=C_TXT))
    a(ParagraphStyle("Toc2", fontName="Helvetica", fontSize=10, leading=18, textColor=C_MUTED, leftIndent=18))
    # Headings
    a(ParagraphStyle("H1", fontName="Helvetica-Bold", fontSize=18, leading=24, textColor=C_NAVY, spaceAfter=8))
    a(ParagraphStyle("H2", fontName="Helvetica-Bold", fontSize=13, leading=18, textColor=C_TEAL, spaceBefore=12, spaceAfter=6))
    # Body
    a(ParagraphStyle("Bd", fontName="Helvetica", fontSize=9.5, leading=13.5, textColor=C_TXT, alignment=TA_JUSTIFY, spaceAfter=4))
    a(ParagraphStyle("BdS", fontName="Helvetica", fontSize=8.5, leading=12, textColor=C_TXT, spaceAfter=2))
    a(ParagraphStyle("Lbl", fontName="Helvetica-Bold", fontSize=9, leading=12, textColor=C_MUTED))
    a(ParagraphStyle("Val", fontName="Helvetica", fontSize=9.5, leading=13, textColor=C_TXT))
    # Table header
    a(ParagraphStyle("Th", fontName="Helvetica-Bold", fontSize=8.5, leading=11, textColor=white))
    a(ParagraphStyle("Td", fontName="Helvetica", fontSize=8.5, leading=12, textColor=C_TXT))
    a(ParagraphStyle("TdB", fontName="Helvetica-Bold", fontSize=8.5, leading=12, textColor=C_TXT))
    # Badges
    a(ParagraphStyle("BgO", fontName="Helvetica-Bold", fontSize=8.5, leading=11, textColor=C_GREEN))
    a(ParagraphStyle("BgN", fontName="Helvetica-Bold", fontSize=8.5, leading=11, textColor=C_RED))
    # Alert
    a(ParagraphStyle("Alt", fontName="Helvetica", fontSize=9.5, leading=14, textColor=C_TXT, leftIndent=14))
    # Doc detail
    a(ParagraphStyle("DocT", fontName="Helvetica-Bold", fontSize=11, leading=15, textColor=C_NAVY, spaceBefore=8, spaceAfter=4))
    # Closing
    a(ParagraphStyle("ClT", fontName="Helvetica-Bold", fontSize=16, leading=22, textColor=white, alignment=TA_CENTER))
    a(ParagraphStyle("ClS", fontName="Helvetica", fontSize=10, leading=14, textColor=HexColor("#8899bb"), alignment=TA_CENTER))
    return s

# ══════════════════════════════════════
# Page callbacks
# ══════════════════════════════════════
def _on_cover(c, doc):
    c.saveState()
    c.setFillColor(C_NAVY)
    c.rect(0, 0, PW, PH, fill=1, stroke=0)
    c.setFillColor(C_ORANGE)
    c.rect(0, PH - 7*mm, PW, 7*mm, fill=1, stroke=0)
    c.setFillColor(C_NAVY_L)
    c.rect(PW - 90*mm, 0, 90*mm, 35*mm, fill=1, stroke=0)
    c.setStrokeColor(C_ORANGE)
    c.setLineWidth(1.5)
    c.line(2.5*cm, 7*cm, PW - 2.5*cm, 7*cm)
    c.restoreState()

def _on_closing(c, doc):
    c.saveState()
    c.setFillColor(C_TEAL)
    c.rect(0, 0, PW, PH, fill=1, stroke=0)
    c.setFillColor(C_ORANGE)
    c.rect(0, 0, PW, 5*mm, fill=1, stroke=0)
    c.restoreState()

def _on_content(c, doc):
    c.saveState()
    # Warm background
    c.setFillColor(HexColor("#FAF8F6"))
    c.rect(0, 0, PW, PH, fill=1, stroke=0)
    # Header line
    c.setStrokeColor(C_ORANGE)
    c.setLineWidth(1)
    c.line(1.5*cm, PH - 1.0*cm, PW - 1.5*cm, PH - 1.0*cm)
    c.setFont("Helvetica", 6.5)
    c.setFillColor(C_MUTED)
    c.drawString(1.5*cm, PH - 0.85*cm, "Gidoo \u2014 Rapport d\u2019analyse RNE")
    dn = getattr(doc, '_dn', '')
    sr = getattr(doc, '_sr', '')
    c.drawRightString(PW - 1.5*cm, PH - 0.85*cm, f"{dn} \u2014 SIREN {sr}")
    # Footer line
    c.setStrokeColor(C_BORDER)
    c.setLineWidth(0.4)
    c.line(1.5*cm, 1.2*cm, PW - 1.5*cm, 1.2*cm)
    c.setFont("Helvetica", 6.5)
    c.setFillColor(C_MUTED)
    rd = getattr(doc, '_rd', '')
    c.drawString(1.5*cm, 0.8*cm, f"Rapport du {rd} \u2014 Source : INPI / Registre National des Entreprises")
    c.drawRightString(PW - 1.5*cm, 0.8*cm, f"Page {c.getPageNumber()}")
    c.restoreState()

# ══════════════════════════════════════
# Alert box (custom Flowable)
# ══════════════════════════════════════
class AlertBox(Flowable):
    def __init__(self, width, paras, title="Points d\u2019attention"):
        Flowable.__init__(self)
        self.bw = width
        self.paras = paras
        self.title = title
        self._pad = 12
        aw = width - 2*self._pad
        self._hs = []
        for p in paras:
            _, h = p.wrap(aw, 10000)
            self._hs.append(h)
        self._th = 26
        self.height = self._th + sum(self._hs) + 2*self._pad + len(paras)*3
        self.width = width

    def draw(self):
        c = self.canv
        p, r = self._pad, 5
        w, h = self.bw, self.height
        c.setFillColor(C_ALERT_BG)
        c.setStrokeColor(C_ORANGE)
        c.setLineWidth(0.5)
        c.roundRect(0, 0, w, h, r, fill=1, stroke=1)
        # Title bar
        pth = c.beginPath()
        pth.moveTo(0, h - self._th)
        pth.lineTo(0, h - r)
        pth.arcTo(0, h-2*r, 2*r, h, startAng=90, extent=90)
        pth.lineTo(w-r, h)
        pth.arcTo(w-2*r, h-2*r, w, h, startAng=0, extent=90)
        pth.lineTo(w, h - self._th)
        pth.close()
        c.setFillColor(C_ORANGE)
        c.drawPath(pth, fill=1, stroke=0)
        c.setFillColor(white)
        c.setFont("Helvetica-Bold", 9.5)
        c.drawString(p, h - self._th + 7, self.title)
        y = h - self._th - p
        aw = w - 2*p
        for para in self.paras:
            _, ph = para.wrap(aw, 10000)
            para.drawOn(c, p, y - ph)
            y -= ph + 3

# ══════════════════════════════════════
# Helpers
# ══════════════════════════════════════
def _x(t):
    """Escape XML for reportlab."""
    if not t: return ""
    return t.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")

def _strip_md(t):
    """Remove markdown bold markers."""
    return re.sub(r'\*\*', '', t) if t else t

def _kv_table(rows, sty, col1=5.5*cm, total_w=None):
    """Build a key-value table with alternating rows."""
    if total_w is None:
        total_w = PW - 3.5*cm
    col2 = total_w - col1
    data = []
    for lbl, val in rows:
        data.append([
            RLPara(f"<b>{_x(lbl)}</b>", sty["Lbl"]),
            RLPara(_x(str(val)), sty["Val"]),
        ])
    t = Table(data, colWidths=[col1, col2])
    ts = [
        ("VALIGN", (0,0), (-1,-1), "MIDDLE"),
        ("TOPPADDING", (0,0), (-1,-1), 5),
        ("BOTTOMPADDING", (0,0), (-1,-1), 5),
        ("LEFTPADDING", (0,0), (-1,-1), 10),
        ("RIGHTPADDING", (0,0), (-1,-1), 10),
        ("LINEBELOW", (0,0), (-1,-2), 0.3, C_BORDER),
    ]
    for i in range(len(data)):
        bg = C_LGRAY if i % 2 == 0 else C_CARD
        ts.append(("BACKGROUND", (0,i), (-1,i), bg))
    t.setStyle(TableStyle(ts))
    return t

def _parse_rne(text):
    """Parse Mistral RNE markdown into structured fields."""
    f = {}
    pats = {
        "denomination": r"[Dd]\u00e9nomination[^:]*:\s*(.+)",
        "siren": r"SIREN[^:]*:\s*(\d{9})",
        "siret": r"SIRET[^:]*:\s*(\d{14})",
        "ape": r"Code\s*APE[^:]*:\s*(.+)",
        "forme": r"Forme\s*juridique[^:]*:\s*(.+)",
        "capital": r"Capital\s*social[^:]*:\s*(.+)",
        "date_immat": r"[Dd]ate\s*d['\u2019]immatriculation[^:]*:\s*(.+)",
        "date_debut": r"[Dd]ate\s*de\s*d[\u00e9e]but\s*d['\u2019]activit[\u00e9e][^:]*:\s*(.+)",
        "duree": r"[Dd]ur[\u00e9e]e[^:]*:\s*(.+)",
        "adresse": r"[Aa]dresse[^:]*:\s*(.+?)(?:\n|$)",
        "code_insee": r"[Cc]ode\s*INSEE[^:]*:\s*(.+)",
        "type_voie": r"[Tt]ype\s*de\s*voie[^:]*:\s*(.+)",
        "dirigeant_nom": r"[Nn]om[^:]*:\s*([A-Z][\w\s]+)",
        "dirigeant_naissance": r"[Dd]ate\s*de\s*naissance[^:]*:\s*(.+)",
        "dirigeant_role": r"[Rr][\u00f4o]le[^:]*:\s*(.+)",
        "nb_etab": r"[Nn]ombre\s*d['\u2019]\u00e9tablissements?\s*ouverts?[^:]*:\s*(.+)",
        "etab_siret": r"\u00c9tablissement\s*principal.*?SIRET[^:]*:\s*(\d{14})",
        "date_creation": r"[Cc]r\u00e9ation[^:]*:\s*(.+)",
        "date_maj": r"[Dd]erni\u00e8re\s*mise\s*\u00e0\s*jour[^:]*:\s*(.+)",
        "date_modif_activite": r"[Dd]erni\u00e8re\s*modification\s*d['\u2019]activit[\u00e9e][^:]*:\s*(.+)",
        "rncs": r"RNCS[^:]*:\s*(.+)",
        "rnm": r"RNM[^:]*:\s*(.+)",
        "raa": r"RAA[^:]*:\s*(.+)",
    }
    for k, p in pats.items():
        m = re.search(p, text)
        if m:
            f[k] = _strip_md(m.group(1).strip().rstrip("."))

    # Objet social (bullet lines only, stop at blank line or next #### section)
    om = re.search(r"[Oo]bjet\s*social[^:]*:\s*\n((?:- .+\n?)+?)(?=\n\n|\n####|\Z)", text)
    if om:
        obj_lines = []
        for l in om.group(1).strip().split("\n"):
            c = re.sub(r"^-\s*", "", _strip_md(l.strip()))
            if c:
                obj_lines.append(c)
        f["objet"] = obj_lines

    # Activités (can be multiple)
    acts = []
    act_blocks = re.findall(
        r"(?:Activit\u00e9[^:]*:.*?\n)((?:.*?(?:Code|Description|Date).*?\n)+)", text, re.IGNORECASE)
    if not act_blocks:
        # Simpler: just grab APE lines
        for m in re.finditer(r"Code\s*APE[^:]*:\s*(.+)", text):
            acts.append(_strip_md(m.group(1).strip()))
    else:
        for block in act_blocks:
            act = {}
            cm = re.search(r"Code\s*APE[^:]*:\s*(.+)", block)
            if cm: act["ape"] = _strip_md(cm.group(1).strip())
            dm = re.search(r"Description[^:]*:\s*(.+)", block)
            if dm: act["desc"] = _strip_md(dm.group(1).strip())
            dtm = re.search(r"Date\s*de\s*d\u00e9but[^:]*:\s*(.+)", block)
            if dtm: act["date"] = _strip_md(dtm.group(1).strip())
            if act: acts.append(act)
    f["activites"] = acts

    # Points d'attention
    pts = []
    pa = re.search(r"[Pp]oints?\s*d['\u2019]attention.*?\n((?:[-*\u2022].*\n?)+)", text)
    if pa:
        for l in pa.group(1).strip().split("\n"):
            c = re.sub(r'^[-*\u2022]\s*', '', _strip_md(l.strip()))
            if c: pts.append(c)
    f["points_attention"] = pts

    return f

# ══════════════════════════════════════
# Main builder
# ══════════════════════════════════════
def generate_report_pdf(siren, denomination, rne_analysis, doc_results, run_date):
    buf = io.BytesIO()
    sty = _styles()

    doc = BaseDocTemplate(buf, pagesize=PAGE,
                          topMargin=1.5*cm, bottomMargin=1.5*cm,
                          leftMargin=1.5*cm, rightMargin=1.5*cm)
    doc._dn = denomination
    doc._sr = siren
    doc._rd = run_date

    cw = PW - 3.0*cm  # content width

    doc.addPageTemplates([
        PageTemplate('cover',
                     frames=[Frame(2.5*cm, 2*cm, PW-5*cm, PH-4*cm, id='cf')],
                     onPage=_on_cover),
        PageTemplate('content',
                     frames=[Frame(1.5*cm, 1.5*cm, cw, PH-3.2*cm, id='ct')],
                     onPage=_on_content),
        PageTemplate('closing',
                     frames=[Frame(2*cm, 2*cm, PW-4*cm, PH-4*cm, id='cl')],
                     onPage=_on_closing),
    ])

    story = []
    _ohx = C_ORANGE.hexval()[2:]
    _mhx = C_MUTED.hexval()[2:]
    _ghx = C_GREEN.hexval()[2:]
    _rhx = C_RED.hexval()[2:]

    rne = _parse_rne(rne_analysis)
    n_docs = len(doc_results)
    n_bilans = sum(1 for d in doc_results if d.get("famille","").upper().startswith("COMP"))
    n_actes = sum(1 for d in doc_results if d.get("famille","").upper() == "ACTE")

    # ────────────────────────────────────────
    # PAGE 1 : COVER
    # ────────────────────────────────────────
    story.append(Spacer(1, 2*cm))
    try:
        story.append(_logo(5.5))
    except Exception:
        pass
    story.append(Spacer(1, 2*cm))
    story.append(RLPara("Rapport d\u2019analyse RNE", sty["CvT"]))
    story.append(Spacer(1, 8*mm))
    story.append(RLPara(_x(denomination), sty["CvS"]))
    story.append(Spacer(1, 3*mm))
    story.append(RLPara(f"SIREN {siren}", sty["CvM"]))
    story.append(Spacer(1, 1.8*cm))
    for ml in [f"Date du rapport : {run_date}",
               f"Documents analys\u00e9s : {n_docs}",
               f"Bilans : {n_bilans}  \u2014  Actes : {n_actes}",
               "Source : Registre National des Entreprises (INPI)"]:
        story.append(RLPara(ml, sty["CvM"]))
        story.append(Spacer(1, 1.5*mm))

    # ────────────────────────────────────────
    # PAGE 2 : SOMMAIRE
    # ────────────────────────────────────────
    story.append(NextPageTemplate('content'))
    story.append(PageBreak())
    story.append(RLPara("Sommaire", sty["TocT"]))
    story.append(HRFlowable(width="25%", thickness=3, color=C_ORANGE,
                            spaceBefore=0, spaceAfter=10*mm, hAlign='LEFT'))

    toc = [
        (1, "A", "Synth\u00e8se entreprise"),
        (2, "A.1", "Identit\u00e9"),
        (2, "A.2", "Objet social"),
        (2, "A.3", "Dirigeant"),
        (2, "A.4", "Adresse du si\u00e8ge"),
        (2, "A.5", "Activit\u00e9s (NAF)"),

        (2, "A.6", "Dates cl\u00e9s"),
        (2, "A.7", "Registres ant\u00e9rieurs"),

        (1, "B", "Analyse des documents d\u00e9pos\u00e9s"),
        (2, "B.1", "Liste des documents"),
        (2, "B.2", "Analyse d\u00e9taill\u00e9e"),
        (1, "C", "Points d\u2019attention"),
    ]
    for level, num, title in toc:
        s = sty["Toc1"] if level == 1 else sty["Toc2"]
        clr = _ohx if level == 1 else _mhx
        story.append(RLPara(
            f"<font color='#{clr}'><b>{num}</b></font>&nbsp;&nbsp;&nbsp;{_x(title)}", s))

    # ────────────────────────────────────────
    # SECTION 1 : SYNTHESE ENTREPRISE
    # ────────────────────────────────────────
    story.append(PageBreak())
    story.append(RLPara(
        f"<font color='#{_ohx}'>A</font>&nbsp;&nbsp;Synth\u00e8se entreprise", sty["H1"]))
    story.append(HRFlowable(width="100%", thickness=1.5, color=C_ORANGE,
                            spaceBefore=0, spaceAfter=6*mm))

    # 1.1 Identité
    story.append(RLPara("A.1&nbsp;&nbsp;Identit\u00e9", sty["H2"]))
    story.append(_kv_table([
        ("D\u00e9nomination", rne.get("denomination", denomination)),
        ("SIREN", rne.get("siren", siren)),
        ("SIRET", rne.get("siret", "\u2014")),
        ("Code APE", rne.get("ape", "\u2014")),
        ("Forme juridique", rne.get("forme", "\u2014")),
        ("Capital social", rne.get("capital", "\u2014")),
        ("Dur\u00e9e", rne.get("duree", "\u2014")),
    ], sty, total_w=cw))

    # A.2 Objet social
    story.append(Spacer(1, 2*mm))
    story.append(RLPara("A.2&nbsp;&nbsp;Objet social", sty["H2"]))
    if rne.get("objet"):
        story.append(_kv_table(
            [(f"Activit\u00e9 {i+1}", line) for i, line in enumerate(rne["objet"])],
            sty, total_w=cw,
        ))
    else:
        story.append(RLPara("Non renseign\u00e9.", sty["Bd"]))

    # A.3 Dirigeant
    story.append(Spacer(1, 2*mm))
    story.append(RLPara("A.3&nbsp;&nbsp;Dirigeant", sty["H2"]))
    story.append(_kv_table([
        ("Nom", rne.get("dirigeant_nom", "\u2014")),
        ("R\u00f4le", rne.get("dirigeant_role", "\u2014")),
        ("Date de naissance", rne.get("dirigeant_naissance", "\u2014")),
    ], sty, total_w=cw))

    # 1.3 Adresse du siège
    story.append(Spacer(1, 2*mm))
    story.append(RLPara("A.4&nbsp;&nbsp;Adresse du si\u00e8ge", sty["H2"]))
    story.append(_kv_table([
        ("Adresse", rne.get("adresse", "\u2014")),
        ("Code INSEE commune", rne.get("code_insee", "\u2014")),
        ("Type de voie", rne.get("type_voie", "\u2014")),
    ], sty, total_w=cw))

    # 1.4 Activités
    story.append(Spacer(1, 2*mm))
    story.append(RLPara("A.5&nbsp;&nbsp;Activit\u00e9s (NAF)", sty["H2"]))
    acts = rne.get("activites", [])
    if acts:
        if isinstance(acts[0], dict):
            act_header = [
                RLPara("<b>Code APE</b>", sty["Th"]),
                RLPara("<b>Description</b>", sty["Th"]),
                RLPara("<b>Date de d\u00e9but</b>", sty["Th"]),
            ]
            act_rows = [act_header]
            for a in acts:
                act_rows.append([
                    RLPara(_x(a.get("ape", "\u2014")), sty["Td"]),
                    RLPara(_x(a.get("desc", "\u2014")), sty["Td"]),
                    RLPara(_x(a.get("date", "\u2014")), sty["Td"]),
                ])
            at = Table(act_rows, colWidths=[4*cm, cw - 4*cm - 3.5*cm, 3.5*cm])
            ats = [
                ("VALIGN", (0,0), (-1,-1), "MIDDLE"),
                ("TOPPADDING", (0,0), (-1,-1), 5),
                ("BOTTOMPADDING", (0,0), (-1,-1), 5),
                ("LEFTPADDING", (0,0), (-1,-1), 8),
                ("BACKGROUND", (0,0), (-1,0), C_NAVY),
                ("TEXTCOLOR", (0,0), (-1,0), white),
                ("LINEBELOW", (0,0), (-1,-1), 0.3, C_BORDER),
            ]
            for i in range(1, len(act_rows)):
                bg = C_LGRAY if i%2==0 else C_CARD
                ats.append(("BACKGROUND", (0,i), (-1,i), bg))
            at.setStyle(TableStyle(ats))
            story.append(at)
        else:
            for a in acts:
                story.append(RLPara(f"\u2022&nbsp;&nbsp;{_x(str(a))}", sty["Bd"]))
    else:
        story.append(RLPara("Aucune activit\u00e9 enregistr\u00e9e.", sty["Bd"]))

    # A.6 Dates clés
    story.append(Spacer(1, 2*mm))
    story.append(RLPara("A.6&nbsp;&nbsp;Dates cl\u00e9s", sty["H2"]))
    story.append(_kv_table([
        ("Date d\u2019immatriculation", rne.get("date_immat", "\u2014")),
        ("D\u00e9but d\u2019activit\u00e9", rne.get("date_debut", "\u2014")),
        ("Derni\u00e8re mise \u00e0 jour", rne.get("date_maj", "\u2014")),
        ("Derni\u00e8re modification d\u2019activit\u00e9", rne.get("date_modif_activite", "\u2014")),
        ("\u00c9tablissements ouverts", rne.get("nb_etab", "\u2014")),
        ("SIRET principal", rne.get("etab_siret", rne.get("siret", "\u2014"))),
    ], sty, total_w=cw))

    # 1.7 Registres antérieurs
    story.append(Spacer(1, 2*mm))
    story.append(RLPara("A.7&nbsp;&nbsp;Registres ant\u00e9rieurs", sty["H2"]))
    story.append(_kv_table([
        ("RNCS", rne.get("rncs", "\u2014")),
        ("RNM", rne.get("rnm", "\u2014")),
        ("RAA", rne.get("raa", "\u2014")),
    ], sty, total_w=cw))

    # ────────────────────────────────────────
    # SECTION B : DOCUMENTS DEPOSES
    # ────────────────────────────────────────
    story.append(PageBreak())
    story.append(RLPara(
        f"<font color='#{_ohx}'>B</font>&nbsp;&nbsp;Analyse des documents d\u00e9pos\u00e9s",
        sty["H1"]))
    story.append(HRFlowable(width="100%", thickness=1.5, color=C_ORANGE,
                            spaceBefore=0, spaceAfter=6*mm))

    # 2.1 Liste des documents (summary table)
    story.append(RLPara("B.1&nbsp;&nbsp;Liste des documents", sty["H2"]))

    if not doc_results:
        story.append(RLPara("Aucun document trouv\u00e9.", sty["Bd"]))
    else:
        hdr = [
            RLPara("<b>#</b>", sty["Th"]),
            RLPara("<b>Date d\u00e9p\u00f4t</b>", sty["Th"]),
            RLPara("<b>Type</b>", sty["Th"]),
            RLPara("<b>Nature INPI</b>", sty["Th"]),
            RLPara("<b>Analys\u00e9</b>", sty["Th"]),
        ]
        rows = [hdr]
        for ix, r in enumerate(doc_results, 1):
            a_sty = sty["BgO"] if r["document_analyse"] == "OUI" else sty["BgN"]
            nature = f"{r.get('typeDocument') or '\u2014'} \u2014 {r.get('typeLibelle') or '\u2014'}"
            rows.append([
                RLPara(str(ix), sty["Td"]),
                RLPara(_x(r.get("date_depot", "\u2014")), sty["Td"]),
                RLPara(_x(r.get("famille", "\u2014")), sty["Td"]),
                RLPara(_x(nature), sty["Td"]),
                RLPara(r["document_analyse"], a_sty),
            ])
        t = Table(rows, colWidths=[1*cm, 3*cm, 3.5*cm, cw - 1*cm - 3*cm - 3.5*cm - 2*cm, 2*cm])
        ts = [
            ("VALIGN", (0,0), (-1,-1), "MIDDLE"),
            ("TOPPADDING", (0,0), (-1,-1), 5),
            ("BOTTOMPADDING", (0,0), (-1,-1), 5),
            ("LEFTPADDING", (0,0), (-1,-1), 6),
            ("BACKGROUND", (0,0), (-1,0), C_NAVY),
            ("TEXTCOLOR", (0,0), (-1,0), white),
            ("LINEBELOW", (0,1), (-1,-1), 0.3, C_BORDER),
        ]
        for i in range(1, len(rows)):
            bg = C_LGRAY if i%2==0 else C_CARD
            ts.append(("BACKGROUND", (0,i), (-1,i), bg))
        t.setStyle(TableStyle(ts))
        story.append(t)

    # 2.2 Analyse détaillée
    story.append(Spacer(1, 6*mm))
    story.append(RLPara("B.2&nbsp;&nbsp;Analyse d\u00e9taill\u00e9e", sty["H2"]))

    for ix, r in enumerate(doc_results, 1):
        story.append(Spacer(1, 4*mm))
        bdg = r["document_analyse"]
        bhx = _ghx if bdg == "OUI" else _rhx
        story.append(RLPara(
            f"<b>Document {ix}/{n_docs}</b>&nbsp;&nbsp;"
            f"<font color='#{bhx}'>[{bdg}]</font>&nbsp;&nbsp;"
            f"{_x(r['filename_base'])}.pdf",
            sty["DocT"]))

        detail_rows = [
            ("Date de d\u00e9p\u00f4t", r.get("date_depot", "\u2014")),
            ("Famille", r.get("famille", "\u2014")),
            ("Nature INPI", f"{r.get('typeDocument') or 'N/A'} \u2014 {r.get('typeLibelle') or 'N/A'}"),
        ]
        dt = _kv_table(detail_rows, sty, col1=4*cm, total_w=cw)
        story.append(dt)

        # Descriptif
        if r.get("descriptif"):
            story.append(Spacer(1, 2*mm))
            story.append(RLPara("<b>Descriptif</b>", sty["Lbl"]))
            dtxt = _strip_md(r["descriptif"].strip())
            for dl in dtxt.split("\n"):
                ds = dl.strip()
                if ds:
                    story.append(RLPara(_x(ds), sty["Bd"]))
        else:
            story.append(RLPara("<i>Aucun descriptif disponible.</i>", sty["BdS"]))

        # Separator
        if ix < n_docs:
            story.append(Spacer(1, 3*mm))
            story.append(HRFlowable(width="100%", thickness=0.3, color=C_BORDER,
                                    spaceBefore=0, spaceAfter=2*mm))

    # ────────────────────────────────────────
    # SECTION 3 : POINTS D'ATTENTION
    # ────────────────────────────────────────
    story.append(PageBreak())
    story.append(RLPara(
        f"<font color='#{_ohx}'>C</font>&nbsp;&nbsp;Points d\u2019attention", sty["H1"]))
    story.append(HRFlowable(width="100%", thickness=1.5, color=C_ORANGE,
                            spaceBefore=0, spaceAfter=6*mm))

    pts = rne.get("points_attention", [])
    if not pts:
        # Fallback: scan text for warning-like lines
        pa = re.findall(
            r'(?:point|anomal|attention|aucun\s*bilan|capital\s*faible|absence|contradiction).*',
            rne_analysis, re.IGNORECASE)
        pts = [re.sub(r'^[-*\u2022]\s*', '', _strip_md(p.strip())) for p in pa if p.strip()]

    if pts:
        apars = [
            RLPara(f"<font color='#{_ohx}'>\u25b8</font>&nbsp;&nbsp;{_x(pt)}", sty["Alt"])
            for pt in pts
        ]
        story.append(AlertBox(cw, apars))
    else:
        story.append(RLPara("Aucun point d\u2019attention identifi\u00e9.", sty["Bd"]))

    # ────────────────────────────────────────
    # CLOSING PAGE
    # ────────────────────────────────────────
    story.append(NextPageTemplate('closing'))
    story.append(PageBreak())
    story.append(Spacer(1, 2.5*cm))
    try:
        lg = _logo(5.5)
        lg.hAlign = 'CENTER'
        story.append(lg)
    except Exception:
        pass
    story.append(Spacer(1, 1.5*cm))
    story.append(RLPara("Automatiser l’analyse RNE dans votre cabinet", sty["ClT"]))
    story.append(Spacer(1, 6*mm))
    story.append(RLPara(
        "Cet agent permet de collecter et structurer automatiquement les données publiques "
        "du Registre National des Entreprises (INPI) afin de produire des rapports exploitables "
        "par les équipes du cabinet.",
        sty["ClS"]))
    story.append(Spacer(1, 5*mm))
    story.append(RLPara("Nous étudions avec vous :", sty["ClS"]))
    story.append(Spacer(1, 2*mm))
    for _blt in [
        "l’intégration dans vos logiciels de production et de gestion interne",
        "l’adaptation de l’agent à vos processus métier",
        "l’automatisation de la collecte et de l’analyse documentaire",
    ]:
        story.append(RLPara(f"\u2022  {_blt}", sty["ClS"]))
    story.append(Spacer(1, 5*mm))
    story.append(RLPara(
        "Les rapports générés peuvent également être personnalisés aux couleurs "
        "de votre cabinet afin d’être utilisés directement dans votre communication client.",
        sty["ClS"]))
    story.append(Spacer(1, 8*mm))
    story.append(RLPara("<b>Contact : contact@iagidoo.fr</b>", sty["ClS"]))

    doc.build(story)
    buf.seek(0)
    return buf.getvalue()

def run_collection(job):
    try:
        job.status = "running"
        job.log("Authentification en cours...", 5)
        token = inpi_login()
        job.log("Authentification reussie", 10)

        # -- Etape 1 : Recuperation donnees entreprise (RNE) --
        job.log(f"Analyse RNE pour SIREN {job.siren}...", 12)
        try:
            company_resp = http_requests.get(
                f"{INPI}/companies/{job.siren}",
                headers={"Authorization": f"Bearer {token}"}, timeout=60
            )
            company_resp.raise_for_status()
            company_json = company_resp.json()
        except Exception as e:
            company_json = {}
            job.log(f"Avertissement : donnees entreprise non disponibles ({e})")

        job.log(f"Recherche des documents pour SIREN {job.siren}...", 15)
        data = inpi_get_attachments(job.siren, token)
        bilans = data.get("bilans", [])
        actes = data.get("actes", [])

        if not bilans and not actes:
            job.log("Aucun document trouve pour ce SIREN", 100)
            job.status = "completed"
            job.errors.append("Aucun document trouve")
            return

        total = len(bilans) + len(actes)
        job.total_docs = total
        job.estimated_minutes = max(1, round(total * 0.35))
        job.log(f"{len(bilans)} bilan(s) et {len(actes)} acte(s) trouves — estimation ~{job.estimated_minutes} min", 20)
        if bilans:
            job.denomination = bilans[0].get("denomination", "")
        elif actes:
            job.denomination = actes[0].get("denomination", "")

        # -- Etape 2 : Analyse RNE via Mistral --
        rne_analysis = ""
        try:
            job.log("Analyse RNE via Mistral AI...", 22)
            rne_prompt = PROMPT_RNE_ANALYSE.format(
                COMPANY_JSON=json.dumps(company_json, ensure_ascii=False)[:15000],
                NB_BILANS=len(bilans),
                NB_ACTES=len(actes),
            )
            rne_analysis = mistral_chat(rne_prompt, max_tokens=1200)
            job.log("Analyse RNE terminee", 25)
        except Exception as e:
            rne_analysis = f"Analyse RNE non disponible : {e}"
            job.log(f"Avertissement analyse RNE : {e}")

        # -- Etape 3 : Telechargement + renommage intelligent + analyse --
        zip_buf = io.BytesIO()
        ppd = 55 / max(total, 1)
        cp = 25
        doc_results = []
        run_date = datetime.now().strftime("%Y-%m-%d")

        with zipfile.ZipFile(zip_buf, 'w', zipfile.ZIP_DEFLATED) as zf:
            for i, b in enumerate(bilans):
                bid = b.get("id")
                if not bid: continue
                cp += ppd
                job.log(f"Bilan {i+1}/{len(bilans)} : telechargement & analyse du document...", int(cp))
                try:
                    content = inpi_dl(f"bilans/{bid}/download", token)
                    res = process_one_document("COMPTES ANNUELS", b, content, job.siren, job.denomination, job)
                    fn = res["filename_base"]
                    zf.writestr(f"COMPTES_ANNUELS/{fn}.pdf", content)
                    doc_results.append(res)
                    job.bilans_count += 1
                    job.current_doc_name = fn
                    job.current_doc_desc = res.get("descriptif", "")[:300]
                    job.log(f"Bilan {i+1}/{len(bilans)} : {fn}")
                except Exception as e:
                    job.errors.append(f"Bilan {bid}: {e}")
                    job.log(f"Erreur bilan {bid}")
                time.sleep(0.3)

            for i, a in enumerate(actes):
                aid = a.get("id")
                if not aid: continue
                ai = inpi_get_acte_info(aid, token)
                merged_meta = {**a, **ai}
                cp += ppd
                job.log(f"Acte {i+1}/{len(actes)} : telechargement et analyse du document...", int(cp))
                try:
                    content = inpi_dl(f"actes/{aid}/download", token)
                    res = process_one_document("ACTE", merged_meta, content, job.siren, job.denomination, job)
                    fn = res["filename_base"]
                    zf.writestr(f"ACTES/{fn}.pdf", content)
                    doc_results.append(res)
                    job.actes_count += 1
                    job.current_doc_name = fn
                    job.current_doc_desc = res.get("descriptif", "")[:300]
                    job.log(f"Acte {i+1}/{len(actes)} : {fn}")
                except Exception as e:
                    job.errors.append(f"Acte {aid}: {e}")
                    job.log(f"Erreur acte {aid}")
                time.sleep(0.3)

            # -- Etape 4 : Generation rapport PDF (Gidoo Premium Landscape) --
            job.log("Generation du rapport PDF Gidoo...", 88)
            report_pdf_bytes = generate_report_pdf(
                job.siren, job.denomination or "Entreprise",
                rne_analysis, doc_results, run_date
            )
            report_filename = _safe_filename(
                f"{run_date} - {job.siren} - {job.denomination or 'Entreprise'} - Rapport Analyse RNE"
            )
            zf.writestr(f"{report_filename}.pdf", report_pdf_bytes)
            job.log(f"Rapport PDF : {report_filename}.pdf")

        zip_buf.seek(0)
        job.zip_data = zip_buf.getvalue()
        job.log(f"{job.bilans_count} bilan(s) + {job.actes_count} acte(s) + rapport PDF prets", 90)
        job.log("✅ Téléchargement prêt", 100)
        job.status = "completed"

    except http_requests.HTTPError as e:
        emsg = f"Erreur API INPI : {e.response.status_code}"
        try: emsg += f" - {e.response.text[:200]}"
        except: pass
        job.log(f"{emsg}", 100)
        job.errors.append(emsg)
        job.status = "error"
    except Exception as e:
        job.log(f"{str(e)}", 100)
        job.errors.append(str(e))
        job.status = "error"

HTML_PAGE = r"""<!DOCTYPE html>
<html lang="fr">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Agent RNE Expert | Gidoo</title>
    <link rel="preconnect" href="https://fonts.googleapis.com">
    <link href="https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;700&family=JetBrains+Mono:wght@400;500&display=swap" rel="stylesheet">
    <style>
        *, *::before, *::after { box-sizing: border-box; margin: 0; padding: 0; }

        :root {
            --gidoo: #EF8829;
            --gidoo-light: #f5a04d;
            --gidoo-glow: rgba(239, 136, 41, 0.08);
            --gidoo-glow-strong: rgba(239, 136, 41, 0.14);
            --gidoo-dark: #d47520;
            --teal: #017e84;
            --teal-light: #e6f5f5;
            --bg: #ffffff;
            --bg-page: #faf8f6;
            --bg-card: #ffffff;
            --bg-input: #f7f6f4;
            --border: #e8e4df;
            --border-focus: #EF8829;
            --text: #2c2c2c;
            --text-muted: #7a7470;
            --text-dim: #a8a29e;
            --success: #2e7d32;
            --error: #c62828;
            --radius: 12px;
            --font: "Inter", -apple-system, BlinkMacSystemFont, sans-serif;
            --mono: "JetBrains Mono", monospace;
        }

        html { scroll-behavior: smooth; }

        body {
            font-family: var(--font);
            background: var(--bg-page);
            color: var(--text);
            min-height: 100vh;
            display: flex;
            flex-direction: column;
            -webkit-font-smoothing: antialiased;
        }

        .wrapper {
            flex: 1; display: flex; flex-direction: column; align-items: center;
            padding: 40px 20px 40px;
        }

        /* ── Header ───────────────────────── */
        .header { text-align: center; margin-bottom: 32px; }

        .header__logo {
            margin-bottom: 16px;
        }
        .header__logo img {
            height: 56px;
            width: auto;
        }

        .header__badge {
            display: inline-block;
            padding: 4px 14px;
            background: var(--gidoo-glow);
            border: 1px solid rgba(239, 136, 41, 0.2);
            border-radius: 20px;
            font-size: 11px; font-weight: 600; color: var(--gidoo);
            letter-spacing: 0.5px; text-transform: uppercase;
            margin-bottom: 18px;
        }

        .header h1 {
            font-size: clamp(22px, 4vw, 30px); font-weight: 700;
            letter-spacing: -0.3px; color: var(--text);
            margin-bottom: 12px; line-height: 1.25;
        }
        .header h1 .line-top {
            display: block; font-size: 0.85em;
            font-weight: 500; color: var(--text-muted);
            letter-spacing: 1.5px; text-transform: uppercase;
            margin-bottom: 4px;
        }
        .header h1 .line-main span { color: var(--gidoo); }

        .header p {
            font-size: 14.5px; color: var(--text-muted);
            max-width: 480px; line-height: 1.65; margin: 0 auto;
        }

        /* ── Card ─────────────────────────── */
        .card {
            width: 100%; max-width: 480px;
            background: var(--bg-card);
            border: 1px solid var(--border);
            border-radius: var(--radius);
            box-shadow: 0 1px 3px rgba(0,0,0,0.03), 0 6px 20px rgba(0,0,0,0.025);
        }

        .card__header {
            padding: 16px 24px;
            border-bottom: 1px solid var(--border);
            font-size: 14px; font-weight: 600; color: var(--text);
            display: flex; align-items: center; gap: 8px;
        }
        .card__header::before {
            content: ''; width: 3px; height: 16px;
            background: var(--gidoo); border-radius: 2px;
        }

        .card__body { padding: 8px; }

        /* ── Form ─────────────────────────── */
        .field { margin-bottom: 18px; }
        .field label {
            display: block; font-size: 13px; font-weight: 500;
            color: var(--text-muted); margin-bottom: 5px;
        }
        .field input {
            width: 100%; padding: 10px 13px;
            background: var(--bg-input); border: 1.5px solid var(--border);
            border-radius: 8px; color: var(--text);
            font-family: var(--mono); font-size: 14.5px;
            transition: all 0.2s; outline: none;
        }
        .field input::placeholder { color: var(--text-dim); }
        .field input:focus {
            border-color: var(--border-focus);
            box-shadow: 0 0 0 3px var(--gidoo-glow);
        }
        .field .hint { font-size: 11.5px; color: var(--text-dim); margin-top: 4px; }

        .btn {
            width: 100%; padding: 11px;
            background: var(--gidoo); color: #fff; border: none;
            border-radius: 8px; font-family: var(--font);
            font-size: 14px; font-weight: 600; cursor: pointer;
            transition: all 0.2s;
        }
        .btn:hover:not(:disabled) {
            background: var(--gidoo-dark);
            box-shadow: 0 2px 12px rgba(239, 136, 41, 0.3);
        }
        .btn:disabled { opacity: 0.5; cursor: not-allowed; }

        .btn--download { background: var(--gidoo); margin-top: 14px; }
        .btn--download:hover:not(:disabled) {
            background: var(--gidoo-dark);
            box-shadow: 0 2px 12px rgba(239, 136, 41, 0.3);
        }

        /* ── Progress ─────────────────────── */
        .progress-section { display: none; margin-top: 20px; }
        .progress-section.active { display: block; }

        .denomination {
            display: none; padding: 10px 14px;
            background: var(--gidoo-glow);
            border: 1px solid rgba(239, 136, 41, 0.18);
            border-radius: 8px; margin-bottom: 14px;
            font-size: 14px; font-weight: 600; color: var(--gidoo-dark);
            text-align: center;
        }
        .denomination.visible { display: block; }

        .progress-info {
            display: flex; justify-content: space-between;
            align-items: center; margin-bottom: 10px;
        }
        .progress-percent {
            font-family: var(--mono); font-size: 13px;
            font-weight: 500; color: var(--gidoo);
        }
        .progress-percent.done { color: var(--success); }
        .progress-percent.error { color: var(--error); }

        .stats { display: flex; gap: 14px; }
        .stat { display: flex; align-items: center; gap: 5px; font-size: 12.5px; color: var(--text-muted); }
        .stat__value { font-family: var(--mono); font-weight: 600; color: var(--text); }

        .progress-bar-wrap {
            height: 5px; background: #ede9e6;
            border-radius: 3px; overflow: hidden; margin-bottom: 14px;
        }
        .progress-bar {
            height: 100%; width: 0%;
            background: linear-gradient(90deg, var(--gidoo), var(--gidoo-light));
            border-radius: 3px; transition: width 0.5s ease;
        }
        .progress-bar.done { background: var(--success); }
        .progress-bar.error { background: var(--error); }

        .log {
            background: #faf9f7; border: 1px solid var(--border);
            border-radius: 8px; padding: 12px;
            max-height: 200px; overflow-y: auto; scroll-behavior: smooth;
        }
        .log::-webkit-scrollbar { width: 3px; }
        .log::-webkit-scrollbar-thumb { background: #d5d0cc; border-radius: 2px; }

        .log__entry {
            font-family: var(--mono); font-size: 11px; line-height: 1.7;
            color: var(--text-muted); padding-left: 8px;
            border-left: 2px solid #e5e0db; margin-bottom: 4px;
            animation: fadeIn 0.3s ease;
        }
        @keyframes fadeIn {
            from { opacity: 0; transform: translateY(3px); }
            to { opacity: 1; transform: translateY(0); }
        }

        /* ── CTA Integration ──────────────── */
        .cta-integration {
            width: 100%; max-width: 480px; margin-top: 24px;
            background: var(--bg-card);
            border: 2px solid var(--gidoo);
            border-radius: var(--radius);
            padding: 22px 24px;
            text-align: center;
            box-shadow: 0 2px 12px rgba(239, 136, 41, 0.1);
        }

        .cta-integration__title {
            font-size: 15px; font-weight: 600; color: var(--text);
            margin-bottom: 6px;
        }

        .cta-integration__sub {
            font-size: 13px; color: var(--text-muted);
            margin-bottom: 14px; line-height: 1.5;
        }

        .cta-integration__btn {
            display: inline-block; padding: 10px 28px;
            background: var(--gidoo); color: #fff; border: none;
            border-radius: 8px; font-family: var(--font);
            font-size: 13px; font-weight: 600; cursor: pointer;
            text-decoration: none; transition: all 0.2s;
        }
        .cta-integration__btn:hover {
            background: var(--gidoo-dark);
            box-shadow: 0 2px 12px rgba(239, 136, 41, 0.3);
        }

        /* ── Contact Form ─────────────────── */
        .contact-form {
            display: none; width: 100%; max-width: 480px;
            background: var(--bg-card);
            border: 1px solid var(--border); border-radius: var(--radius);
            padding: 24px; margin-top: 12px;
            box-shadow: 0 1px 3px rgba(0,0,0,0.04);
        }
        .contact-form.open { display: block; }
        .contact-form h3 { font-size: 15px; margin-bottom: 14px; color: var(--text); }

        .contact-form textarea {
            width: 100%; padding: 10px 13px;
            background: var(--bg-input); border: 1.5px solid var(--border);
            border-radius: 8px; color: var(--text); font-family: var(--font);
            font-size: 13px; resize: vertical; min-height: 80px;
            outline: none; transition: border-color 0.2s;
        }
        .contact-form textarea:focus {
            border-color: var(--border-focus);
            box-shadow: 0 0 0 3px var(--gidoo-glow);
        }

        /* ── Footer ───────────────────────── */
        .footer {
            text-align: center; padding: 20px 20px 28px;
        }
        .footer__brand {
            margin-bottom: 8px;
        }
        .footer__brand a {
            display: inline-flex; align-items: center; gap: 6px;
            text-decoration: none; font-weight: 600; font-size: 14px;
            color: var(--gidoo);
        }
        .footer__brand a:hover { text-decoration: underline; }
        .footer__brand img { height: 24px; width: auto; }

        .footer__tagline {
            font-size: 12.5px; color: var(--text-muted); margin-bottom: 4px;
        }
        .footer__legal {
            font-size: 11px; color: var(--text-dim);
        }

        /* ── Toast ────────────────────────── */
        .toast {
            position: fixed; bottom: 24px; left: 50%;
            transform: translateX(-50%) translateY(80px);
            padding: 12px 24px; background: #fff;
            border: 1px solid var(--border); border-radius: 10px;
            font-size: 14px; color: var(--text); opacity: 0;
            box-shadow: 0 4px 20px rgba(0,0,0,0.08);
            transition: all 0.4s cubic-bezier(0.16, 1, 0.3, 1); z-index: 100;
        }
        .toast.show { opacity: 1; transform: translateX(-50%) translateY(0); }
        .toast.success { border-color: var(--success); }
        .toast.error { border-color: var(--error); }

        @media (max-width: 540px) {
            .card__body { padding: 20px; }
            .stats { flex-direction: column; gap: 5px; }
            .cta-integration { padding: 18px 16px; }
        }
    </style>
</head>
<body>

<div class="wrapper">
    <header class="header">
        <div class="header__logo">
            <img src="..." alt="gidoo">
        </div>
        <div class="header__badge">Service gratuit</div>
        <h1>
            <span class="line-top">Agent</span>
            <span class="line-main"><span>RNE</span> <span>Expert</span></span>
        </h1>

        <p class="header__sub">Collectez, analysez et synthétisez l'ensemble des comptes annuels et actes juridiques déposés par une entreprise auprès de l'INPI. L'agent renomme chaque document selon la nomenclature officielle, en extrait le contenu et produit un rapport d'analyse complet</p>
</header>

    <div class="card">
        <div class="card__header">Renseignez un SIREN et lancer l'analyse</div>
        <div class="card__body">
            <form id="collectForm">
                <div class="field">
                    <label for="siren">Numéro SIREN</label>
                    <input type="text" id="siren" placeholder="123 456 789" maxlength="11" autocomplete="off" required>
                    <div class="hint">9 chiffres : identifiant unique de l'entreprise</div>
                </div>
<button type="submit" class="btn" id="submitBtn">Lancer l'analyse</button>
            </form>

            <div class="progress-section" id="progressSection">
                <div class="denomination" id="denomination"></div>
                <div class="progress-info">
                    <div class="stats">
                        <div class="stat">📊 <span class="stat__value" id="bilansCount">0</span> bilans</div>
                        <div class="stat">📑 <span class="stat__value" id="actesCount">0</span> actes</div>
                    </div>
                    <div class="progress-percent" id="progressPercent">0%</div>
                </div>
                <div class="progress-bar-wrap">
                    <div class="progress-bar" id="progressBar"></div>
                </div>
                <!-- Estimation temps -->
                <div id="estimatedTime" style="display:none; margin-bottom:12px; padding:10px 14px; background:#f0fdf4; border:1px solid #bbf7d0; border-radius:8px; font-size:13px; color:#166534; text-align:center;">
                    ⏱️ Temps estimé : <strong id="estMinutes">—</strong> min
                </div>

                <!-- Document analysé -->
                <div id="currentDocBlock" style="display:none; margin-bottom:14px; padding:14px 16px; background:var(--gidoo-glow); border:1px solid rgba(239, 136, 41, 0.18); border-radius:8px;">
                    <div style="font-size:11px; font-weight:600; text-transform:uppercase; letter-spacing:0.5px; color:var(--gidoo); margin-bottom:6px;">📄 Document analysé</div>
                    <div id="currentDocName" style="font-family:var(--mono); font-size:12.5px; font-weight:600; color:var(--text); word-break:break-word; margin-bottom:6px;"></div>
                    <div id="currentDocDesc" style="font-size:12px; color:var(--text-muted); line-height:1.55;"></div>
                </div>

                <div class="log" id="logContainer"></div>
                <button class="btn btn--download" id="downloadBtn" style="display:none">Télécharger le ZIP</button>
                <button class="btn" id="resetBtn" style="display:none; margin-top:10px; background:#a8a29e;">Nouvelle collecte</button>
            </div>
        </div>
    </div>

    <!-- CTA Intégration — bien visible -->

<div style="max-width:480px;margin:8px auto 10px auto;text-align:center;font-size:14px;color:#6b7280;line-height:1.6;">

<div class="cta-integration">
        <div class="cta-integration__title">
            Intégrer cet agent dans votre Cabinet ?
        </div>
        <div class="cta-integration__sub">
            Nous étudions l'intégration adaptée à votre cabinet, à vos logiciels de Production et de Gestion Interne.
        </div>
        <button class="cta-integration__btn" id="contactToggle">Contactez-nous</button>
    </div>

    <div class="contact-form" id="contactForm">
        <h3>Intégration sur mesure</h3>
        <div class="field">
            <label for="contactName">Nom / Cabinet</label>
            <input type="text" id="contactName" placeholder="Votre nom ou cabinet">
        </div>
        <div class="field">
            <label for="contactEmail">Email</label>
            <input type="email" id="contactEmail" placeholder="vous@cabinet.fr">
        </div>
        <div class="field">
            <label for="contactPhone">Téléphone mobile</label>
            <input type="tel" id="contactPhone" placeholder="06 12 34 56 78">
        </div>
        <div class="field">
            <label for="contactMessage">Vos besoins</label>
            <textarea id="contactMessage" placeholder="Volume estimé, outils utilisés, nombre de collaborateurs..."></textarea>
        </div>
        <button class="btn" id="contactSubmit">Envoyer</button>
    </div>
</div>

<footer class="footer">
    <div class="footer__brand">
        <a href="https://www.gidoo.fr" target="_blank">
            <img src="..." alt="gidoo">
        </a>
        <div style="margin-top: 4px;"><a href="https://www.gidoo.fr" target="_blank" style="font-size: 12.5px; color: var(--text-muted); text-decoration: none;">www.gidoo.fr</a></div>
    </div>
    <div class="footer__tagline">Conçu pour les Cabinets d'Expertise Comptable</div>
    <div class="footer__legal">Données issues de sources officielles</div>
</footer>

<div class="toast" id="toast"></div>

<script>
(function() {
    const form = document.getElementById("collectForm");
    const sirenInput = document.getElementById("siren");
    const submitBtn = document.getElementById("submitBtn");
    const progressSection = document.getElementById("progressSection");
    const progressBar = document.getElementById("progressBar");
    const progressPercent = document.getElementById("progressPercent");
    const bilansCount = document.getElementById("bilansCount");
    const actesCount = document.getElementById("actesCount");
    const logContainer = document.getElementById("logContainer");
    const downloadBtn = document.getElementById("downloadBtn");
    const resetBtn = document.getElementById("resetBtn");
    const denomination = document.getElementById("denomination");
    const toast = document.getElementById("toast");
    const estimatedTimeBlock = document.getElementById("estimatedTime");
    const estMinutes = document.getElementById("estMinutes");
    const currentDocBlock = document.getElementById("currentDocBlock");
    const currentDocName = document.getElementById("currentDocName");
    const currentDocDesc = document.getElementById("currentDocDesc");
    let currentJobId = null;

    sirenInput.addEventListener("input", function(e) {
        let v = e.target.value.replace(/\D/g, "").slice(0, 9);
        if (v.length > 6) v = v.slice(0,3)+" "+v.slice(3,6)+" "+v.slice(6);
        else if (v.length > 3) v = v.slice(0,3)+" "+v.slice(3);
        e.target.value = v;
    });

    function showToast(msg, type) {
        toast.textContent = msg;
        toast.className = "toast " + (type || "");
        requestAnimationFrame(() => toast.classList.add("show"));
        setTimeout(() => toast.classList.remove("show"), 4000);
    }

    function addLog(msg) {
        const d = document.createElement("div");
        d.className = "log__entry"; d.textContent = msg;
        logContainer.appendChild(d);
        logContainer.scrollTop = logContainer.scrollHeight;
    }

    function startPolling(jobId) {
        const poll = setInterval(async () => {
            try {
                const r = await fetch("/api/status/" + jobId);
                const d = await r.json();
                const last = parseInt(logContainer.dataset.lastStep || "0");
                if (d.steps && d.steps.length > last) {
                    for (let i = last; i < d.steps.length; i++) addLog(d.steps[i].message);
                    logContainer.dataset.lastStep = d.steps.length;
                }
                progressBar.style.width = d.progress + "%";
                progressPercent.textContent = d.progress + "%";
                bilansCount.textContent = d.bilans_count || 0;
                actesCount.textContent = d.actes_count || 0;
                if (d.denomination) {
                    denomination.textContent = d.denomination;
                    denomination.classList.add("visible");
                }
                if (d.estimated_minutes > 0) {
                    estMinutes.textContent = d.estimated_minutes;
                    estimatedTimeBlock.style.display = "block";
                }
                if (d.current_doc_name) {
                    currentDocName.textContent = d.current_doc_name;
                    currentDocDesc.textContent = d.current_doc_desc || "";
                    currentDocBlock.style.display = "block";
                }
                if (d.status === "completed" || d.status === "error") {
                    clearInterval(poll);
                    if (d.status === "completed" && (d.bilans_count > 0 || d.actes_count > 0)) {
                        progressBar.classList.add("done");
                        progressPercent.classList.add("done");
                        downloadBtn.style.display = "block";
                        currentDocBlock.style.display = "none";
                        estimatedTimeBlock.style.display = "none";
                        showToast("✅ Collecte terminée. Cliquez pour télécharger le ZIP", "success");
                    } else if (d.status === "error") {
                        progressBar.classList.add("error");
                        progressPercent.classList.add("error");
                        showToast("Erreur : " + (d.errors[0] || ""), "error");
                    } else {
                        progressBar.classList.add("error");
                        showToast("Aucun document trouvé pour ce SIREN", "error");
                    }
                    resetBtn.style.display = "block";
                }
            } catch(e) { console.error(e); }
        }, 1500);
    }

    form.addEventListener("submit", async function(e) {
        e.preventDefault();
        const siren = sirenInput.value.replace(/\s/g, "");
if (!/^\d{9}$/.test(siren)) { showToast("SIREN : 9 chiffres requis", "error"); return; }
        submitBtn.disabled = true; submitBtn.textContent = "Lancement...";
        try {
            const r = await fetch("/api/collect", {
                method: "POST", headers: {"Content-Type": "application/json"},
                body: JSON.stringify({siren})
            });
            const d = await r.json();
            if (!r.ok) { showToast(d.error, "error"); submitBtn.disabled=false; submitBtn.textContent="Lancer la collecte"; return; }
            currentJobId = d.job_id;
            form.style.display = "none";
            progressSection.classList.add("active");
            logContainer.dataset.lastStep = "0";
            startPolling(currentJobId);
        } catch(err) {
            showToast("Erreur réseau", "error");
            submitBtn.disabled=false; submitBtn.textContent="Lancer la collecte";
        }
    });

    downloadBtn.addEventListener("click", () => { if(currentJobId) window.location.href="/api/download/"+currentJobId; });

    resetBtn.addEventListener("click", function() {
        form.style.display="block"; progressSection.classList.remove("active");
        progressBar.style.width="0%"; progressBar.className="progress-bar";
        progressPercent.className="progress-percent"; progressPercent.textContent="0%";
        bilansCount.textContent="0"; actesCount.textContent="0";
        logContainer.innerHTML=""; downloadBtn.style.display="none";
        resetBtn.style.display="none"; denomination.classList.remove("visible");
        submitBtn.disabled=false; submitBtn.textContent="Lancer la collecte";
        sirenInput.value=""; currentJobId=null;
    });

    document.getElementById("contactToggle").addEventListener("click", () => {
        document.getElementById("contactForm").classList.toggle("open");
    });

    document.getElementById("contactSubmit").addEventListener("click", async () => {
        const name=document.getElementById("contactName").value.trim();
        const email=document.getElementById("contactEmail").value.trim();
        const phone=document.getElementById("contactPhone").value.trim();
        const message=document.getElementById("contactMessage").value.trim();
        if(!name||!email||!phone||!message){showToast("Tous les champs sont requis","error");return;}
        const emailRegex=/^[^\s@]+@[^\s@]+\.[^\s@]+$/;
        if(!emailRegex.test(email)){showToast("Adresse email invalide","error");return;}
        const phoneClean=phone.replace(/[\s.\-()]/g,"");
        const phoneRegex=/^(0[67]\d{8}|\+33[67]\d{8})$/;
        if(!phoneRegex.test(phoneClean)){showToast("Téléphone mobile invalide (06/07 ou +336/+337)","error");return;}
        try {
            const r=await fetch("/api/contact",{method:"POST",headers:{"Content-Type":"application/json"},body:JSON.stringify({name,email,phone,message})});
            if(r.ok){showToast("Message envoyé !","success");document.getElementById("contactForm").classList.remove("open");}
            else showToast("Erreur","error");
        }catch(e){showToast("Erreur réseau","error");}
    });
})();
</script>
</body>
</html>

"""
@app.route("/")
def index():
    return HTML_PAGE

@app.route("/api/collect", methods=["POST"])
def api_collect():
    data = flask_request.json or {}
    siren = re.sub(r'\s+', '', data.get("siren", ""))
    if not re.match(r'^\d{9}$', siren):
        return jsonify({"error": "SIREN : 9 chiffres requis"}), 400

    job = Job(siren)
    jobs[job.id] = job
    Thread(target=run_collection, args=(job,), daemon=True).start()
    return jsonify({"job_id": job.id})

@app.route("/api/status/<job_id>")
def api_status(job_id):
    job = jobs.get(job_id)
    if not job: return jsonify({"error": "Job non trouve"}), 404
    return jsonify(job.to_dict())

@app.route("/api/download/<job_id>")
def api_download(job_id):
    job = jobs.get(job_id)
    if not job or not job.zip_data:
        return jsonify({"error": "Aucun document"}), 404
    buf = io.BytesIO(job.zip_data)
    buf.seek(0)
    fn = sanitize(f"{job.siren} - {datetime.now().strftime('%Y-%m-%d')}") + ".zip"
    return send_file(buf, mimetype="application/zip", as_attachment=True, download_name=fn)

@app.route("/api/contact", methods=["POST"])
def api_contact():
    data = flask_request.json or {}
    name = data.get("name", "").strip()
    email = data.get("email", "").strip()
    phone = data.get("phone", "").strip()
    message = data.get("message", "").strip()
    if not all([name, email, phone, message]):
        return jsonify({"error": "Champs requis"}), 400
    email_regex = re.compile(r'^[^\s@]+@[^\s@]+\.[^\s@]+$')
    if not email_regex.match(email):
        return jsonify({"error": "Adresse email invalide"}), 400
    phone_clean = re.sub(r'[\s.\-()]', '', phone)
    phone_regex = re.compile(r'^(0[67]\d{8}|\+33[67]\d{8})$')
    if not phone_regex.match(phone_clean):
        return jsonify({"error": "Téléphone mobile invalide (06/07 ou +336/+337)"}), 400
    ok = send_contact_email(name, email, phone, message)
    if ok:
        return jsonify({"message": "Envoyé"})
    else:
        return jsonify({"message": "Reçu", "warning": "Envoi email non confirmé (réseau/SMTP)"}), 202

@app.route("/health")
def health():
    return jsonify({"status": "ok"})

print("Application Gidoo chargee")
print("   Email : fallback SSL:465 > STARTTLS:587 > P25")

Application Gidoo chargee
   Email : fallback SSL:465 > STARTTLS:587 > P25


## 4️⃣ Tunnel public (Cloudflared)

1) Installe cloudflared
2) Démarre Flask puis ouvre le tunnel

👉 Le lien `https://…trycloudflare.com` apparaît dans la sortie de la dernière cellule.

In [ ]:
# Cloudflared (tunnel public) - installation
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
!cloudflared --version
print("✅ cloudflared prêt")

cloudflared version 2026.2.0 (built 2026-02-06-14:47 UTC)
✅ cloudflared prêt


In [ ]:
import threading, time

# 1️⃣ Démarrer Flask en arrière-plan (non bloquant)
def run_flask():
    app.run(host="0.0.0.0", port=5000, debug=False, use_reloader=False)

flask_thread = threading.Thread(target=run_flask, daemon=True)
flask_thread.start()

time.sleep(2)
print("✅ Flask démarré sur http://127.0.0.1:5000")

# 2️⃣ Ouvrir le tunnel Cloudflare (bloquant) : le lien trycloudflare.com s'affiche ci-dessous
# ⚠️ Laisse cette cellule tourner.
!cloudflared tunnel --url http://127.0.0.1:5000

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.28.0.12:5000
INFO:werkzeug:Press CTRL+C to quit


✅ Flask démarré sur http://127.0.0.1:5000
2026-03-05T07:59:47Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-03-05T07:59:47Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-03-05T07:59:50Z INF +--------------------------------------------------------------------------------------------+
2026-03-05T07:59:50Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-03-05T07:59:50Z INF |  https://hap

INFO:werkzeug:127.0.0.1 - - [05/Mar/2026 07:59:55] "GET / HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [05/Mar/2026 08:00:02] "GET /favicon.ico HTTP/1.1" 404 -
INFO:werkzeug:127.0.0.1 - - [05/Mar/2026 08:00:14] "POST /api/collect HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [05/Mar/2026 08:00:17] "GET /api/status/2232cb1a HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [05/Mar/2026 08:00:18] "GET /api/status/2232cb1a HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [05/Mar/2026 08:00:19] "GET /api/status/2232cb1a HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [05/Mar/2026 08:00:21] "GET /api/status/2232cb1a HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [05/Mar/2026 08:00:22] "GET /api/status/2232cb1a HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [05/Mar/2026 08:00:24] "GET /api/status/2232cb1a HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [05/Mar/2026 08:00:25] "GET /api/status/2232cb1a HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [05/Mar/2026 08:00:27] "GET /api/status/2232cb1a HTTP/1.1" 200 -
INFO:werkze